In [1]:
# Cell 1 - Environment setup, directory contract, and OJK LPBBTI catalog discovery
from pathlib import Path
import platform
import random
import re
import numpy as np
import pandas as pd
import openpyxl
import curl_cffi
from bs4 import BeautifulSoup
from curl_cffi import requests

SEED = 42
FORCE_REFRESH = False
random.seed(SEED)
np.random.seed(SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
assert (PROJECT_ROOT / "README.md").exists(), f"Repository root not found from {cwd}"

DIRS = {
    "raw_ojk_xlsx": PROJECT_ROOT / "data" / "raw" / "ojk_xlsx",
    "raw_ojk_sjkpublic": PROJECT_ROOT / "data" / "raw" / "ojk_sjkpublic",
    "raw_ojk_directory": PROJECT_ROOT / "data" / "raw" / "ojk_directory",
    "raw_bps": PROJECT_ROOT / "data" / "raw" / "bps",
    "raw_playstore": PROJECT_ROOT / "data" / "raw" / "playstore",
    "clean": PROJECT_ROOT / "data" / "clean",
    "artifacts": PROJECT_ROOT / "artifacts",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(
    f"step=env_setup | python={platform.python_version()} | pandas={pd.__version__} | "
    f"curl_cffi={curl_cffi.__version__} | openpyxl={openpyxl.__version__} | seed={SEED}"
)
print(f"project_root={PROJECT_ROOT.name} | dirs_ready={len(DIRS)}")

session = requests.Session(impersonate="chrome")

# 1. Discover historical monthly OJK Statistik LPBBTI pages across SharePoint pager pages 1..5
base_ojk = "https://ojk.go.id"
old_url = f"{base_ojk}/id/kanal/iknb/data-dan-statistik/fintech/olddefault.aspx"
r0 = session.get(old_url, timeout=30)
r0.raise_for_status()

def parse_ojk_page(html_text):
    soup = BeautifulSoup(html_text, "lxml")
    rows = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        title = a.get_text(strip=True)
        if "statistik-lpbbti" in href.lower() and href.lower().endswith(".aspx"):
            full_url = href if href.startswith("http") else base_ojk + href
            rows.append((title, full_url))
    return soup, rows

soup, first_rows = parse_ojk_page(r0.text)
historical_pages = list(dict.fromkeys(first_rows))

for target_page in range(2, 6):
    pager_link = None
    for a in soup.find_all("a", href=True):
        if a.get_text(strip=True) == str(target_page) and "DataPagerArticles" in a["href"]:
            pager_link = a["href"]
            break
    if not pager_link:
        break
    m = re.search(r"__doPostBack\('([^']+)','([^']*)'\)", pager_link)
    if not m:
        break
    payload = {inp.get("name"): inp.get("value", "") for inp in soup.find_all("input") if inp.get("name")}
    payload["__EVENTTARGET"] = m.group(1)
    payload["__EVENTARGUMENT"] = m.group(2)
    rp = session.post(old_url, data=payload, timeout=30)
    rp.raise_for_status()
    soup, next_rows = parse_ojk_page(rp.text)
    for item in next_rows:
        if item not in historical_pages:
            historical_pages.append(item)

df_ojk_hist = pd.DataFrame(historical_pages, columns=["title", "page_url"])
print(f"ojk_historical_pages={len(df_ojk_hist)}")
print(df_ojk_hist.head(5).to_string(index=False))
print(df_ojk_hist.tail(5).to_string(index=False))

# 2. Discover OJK SJKPublic LPBBTI Konvensional (18) and Syariah (42) dataset metrics
sjk_base = "https://data.ojk.go.id/SJKPublic/Dataset/Dataset"
session.get(sjk_base, timeout=30).raise_for_status()
sjk_rows = []
for ind_id, ind_label in [(18, "LPBBTI_Konvensional"), (42, "LPBBTI_Syariah")]:
    info_resp = session.get(f"{sjk_base}/GetInformasiData", params={"industriId": ind_id}, timeout=30).json()
    for info in info_resp.get("data", []):
        m_resp = session.get(
            f"{sjk_base}/GetMetricData",
            params={"informasiId": info["InformasiID"]},
            timeout=30,
        ).json()
        for m in m_resp.get("data", []):
            sjk_rows.append({
                "industri_id": ind_id,
                "industri": ind_label,
                "informasi_id": info["InformasiID"],
                "informasi": info["InformasiName"],
                "metric_id": m["MetricID"],
                "metric_name": m["MetricName"],
            })

df_sjk_catalog = pd.DataFrame(sjk_rows)
print(f"sjkpublic_metrics_count={len(df_sjk_catalog)}")
print(df_sjk_catalog[["industri", "informasi", "metric_id", "metric_name"]].to_string(index=False))


step=env_setup | python=3.14.6 | pandas=3.0.3 | curl_cffi=0.16.3 | openpyxl=3.1.5 | seed=42
project_root=pindar-risk-atlas | dirs_ready=7


ojk_historical_pages=10
                                                                                                            title                                                                                                                      page_url
PENGUMUMAN: Peralihan Publikasi Statistik Layanan Pendanaan Bersama Berbasis Teknologi Informasi (LPBBTI)/Fintech https://ojk.go.id/id/kanal/iknb/data-dan-statistik/fintech/Pages/PENGUMUMAN-Peralihan-Publikasi-Statistik-LPBBTI-Fintech.aspx
                                                                                   Statistik LPBBTI Desember 2025                          https://ojk.go.id/id/kanal/iknb/data-dan-statistik/fintech/Pages/Statistik-LPBBTI-Desember-2025.aspx
                                                                                   Statistik LPBBTI November 2025                          https://ojk.go.id/id/kanal/iknb/data-dan-statistik/fintech/Pages/Statistik-LPBBTI-November-2025.aspx
                

sjkpublic_metrics_count=56
           industri                informasi  metric_id                                                           metric_name
LPBBTI_Konvensional     Inf. Entitas Pelapor        135                                             Jumlah Entitas Izin Aktif
LPBBTI_Konvensional          Informasi Acuan        416                       Inf. Kontrak Usaha dan Rekening - DATI1 Pemberi
LPBBTI_Konvensional          Informasi Acuan        417                      Inf. Kontrak Usaha dan Rekening - DATI1 Penerima
LPBBTI_Konvensional          Informasi Acuan        418                      Inf. Kontrak Usaha dan Rekening - Jenis Industri
LPBBTI_Konvensional          Informasi Acuan        419                    Inf. Kontrak Usaha dan Rekening - Kategori Pemberi
LPBBTI_Konvensional          Informasi Acuan        420                   Inf. Kontrak Usaha dan Rekening - Kategori Penerima
LPBBTI_Konvensional          Informasi Acuan        421                     Inf. Kontrak Us

In [2]:
# Cell 2 - Extract OJK SJKPublic LPBBTI datasets and Mapping Statistik LPBBTI.xlsx
import time
import requests as std_requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

TARGET_METRICS = [
    (144, "konv_twp90_dati1"),
    (312, "syar_twp90_dati1"),
    (411, "konv_borrower_dati1"),
    (427, "syar_borrower_dati1"),
    (1453, "konv_borrower_kualitas_dati1"),
    (1454, "syar_borrower_kualitas_dati1"),
    (407, "konv_lender_dati1"),
    (423, "syar_lender_dati1"),
    (414, "konv_borrower_usia"),
    (430, "syar_borrower_usia"),
    (415, "konv_borrower_sektor"),
    (431, "syar_borrower_sektor"),
    (417, "konv_rekening_dati1"),
    (433, "syar_rekening_dati1"),
    (384, "konv_rasio_keuangan"),
    (316, "syar_rasio_keuangan"),
    (444, "konv_twp90_gender"),
    (445, "syar_twp90_gender"),
]

http_sess = std_requests.Session()
retries = Retry(total=4, connect=4, read=4, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET", "POST"])
http_sess.mount("https://", HTTPAdapter(max_retries=retries))
http_sess.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
})

# Download Mapping Statistik LPBBTI.xlsx reference document
mapping_url = "https://ojk.go.id/id/kanal/iknb/data-dan-statistik/fintech/Documents/Mapping%20Statistik%20LPBBTI.xlsx"
mapping_path = DIRS["raw_ojk_sjkpublic"] / "Mapping_Statistik_LPBBTI.xlsx"
if FORCE_REFRESH or not (mapping_path.exists() and mapping_path.stat().st_size > 1000):
    r_map = http_sess.get(mapping_url, timeout=60)
    r_map.raise_for_status()
    mapping_path.write_bytes(r_map.content)
print(f"mapping_file={mapping_path.name} | bytes={mapping_path.stat().st_size}")

# Initialize XSRF token on SJKPublic
sjk_base = "https://data.ojk.go.id/SJKPublic/Dataset/Dataset"
r_home = http_sess.get(sjk_base, timeout=30)
r_home.raise_for_status()
xsrf_soup = BeautifulSoup(r_home.text, "lxml")
xsrf_token = xsrf_soup.find("input", {"name": "__RequestVerificationToken"})["value"]

sjk_summary = []
for metric_id, slug in TARGET_METRICS:
    cached_matches = sorted(DIRS["raw_ojk_sjkpublic"].glob(f"{metric_id}_{slug}_*.csv"))
    if not FORCE_REFRESH and cached_matches:
        out_csv = cached_matches[0]
        nama_table = out_csv.stem.replace(f"{metric_id}_{slug}_", "", 1)
        df_tmp = pd.read_csv(out_csv)
        bulan_vals = sorted(df_tmp["Bulan"].dropna().astype(str).unique()) if "Bulan" in df_tmp.columns else []
        n_prov = df_tmp["NamaDATI1"].nunique() if "NamaDATI1" in df_tmp.columns else 0
        sjk_summary.append({
            "metric_id": metric_id, "slug": slug, "nama_table": nama_table,
            "rows": len(df_tmp), "cols": len(df_tmp.columns),
            "bulan_min": bulan_vals[0] if bulan_vals else "NONE",
            "bulan_max": bulan_vals[-1] if bulan_vals else "NONE",
            "n_months": len(bulan_vals), "n_prov": n_prov,
        })
        continue
    gm = http_sess.get(f"{sjk_base}/GetGridMeasure", params={"refMetricId": metric_id}, timeout=30).json()
    measures = ",".join([item["name"] for item in gm.get("data", [])])

    r_save = http_sess.post(
        f"{sjk_base}/SaveMeasures",
        headers={"XSRF-TOKEN": xsrf_token},
        data={"measures": measures, "id": metric_id},
        timeout=30,
    )
    r_save.raise_for_status()

    r_ds = http_sess.get(f"{sjk_base}/Dataset/{metric_id}", timeout=30)
    r_ds.raise_for_status()
    m_table = re.search(r'"namaTable"\s*:\s*"([^"]+)"', r_ds.text)
    assert m_table, f"namaTable not found for metric_id={metric_id}"
    nama_table = m_table.group(1)

    r_grid = http_sess.get(
        f"{sjk_base}/GetGridCSVData",
        params={"_filter": "", "_field": "", "_data": "", "namaTable": nama_table, "metricID": metric_id},
        timeout=60,
    )
    r_grid.raise_for_status()
    payload = r_grid.json()
    rows = payload.get("data", [])
    df_tmp = pd.DataFrame(rows)

    out_csv = DIRS["raw_ojk_sjkpublic"] / f"{metric_id}_{slug}_{nama_table}.csv"
    df_tmp.to_csv(out_csv, index=False)

    bulan_vals = sorted(df_tmp["Bulan"].dropna().astype(str).unique()) if "Bulan" in df_tmp.columns else []
    n_prov = df_tmp["NamaDATI1"].nunique() if "NamaDATI1" in df_tmp.columns else 0
    sjk_summary.append({
        "metric_id": metric_id,
        "slug": slug,
        "nama_table": nama_table,
        "rows": len(df_tmp),
        "cols": len(df_tmp.columns),
        "bulan_min": bulan_vals[0] if bulan_vals else "NONE",
        "bulan_max": bulan_vals[-1] if bulan_vals else "NONE",
        "n_months": len(bulan_vals),
        "n_prov": n_prov,
    })
    time.sleep(0.4)

df_sjk_extracted = pd.DataFrame(sjk_summary)
print(f"sjk_extracted_tables={len(df_sjk_extracted)} | total_rows={df_sjk_extracted['rows'].sum()}")
print(df_sjk_extracted.to_string(index=False))


mapping_file=Mapping_Statistik_LPBBTI.xlsx | bytes=23507


sjk_extracted_tables=18 | total_rows=18223
 metric_id                         slug                                              nama_table  rows  cols bulan_min bulan_max  n_months  n_prov
       144             konv_twp90_dati1                                      LPBBTI_TWP90_DATI1   880     9   2024-07   2026-04        22      40
       312             syar_twp90_dati1                              LPBBTI_Syariah_TWP90_DATI1   880     9   2024-07   2026-04        22      40
       411          konv_borrower_dati1                            LPBBTI_PenerimaPinjamanDATI1   902    11   2024-07   2026-04        22      41
       427          syar_borrower_dati1                    LPBBTI_Syariah_PenerimaPinjamanDATI1   902    11   2024-07   2026-04        22      41
      1453 konv_borrower_kualitas_dati1                  LPBBTI_PenerimaPinjamanKualitasPerDati  4180    13   2024-07   2026-04        22       0
      1454 syar_borrower_kualitas_dati1          LPBBTI_Syariah_PenerimaPinjamanK

In [3]:
# Cell 3 - Download OJK historical XLSX workbooks (2023-2025) and parse OJK Direktori LPBBTI PDF
import pdfplumber

MONTHS_ID = [
    "Januari", "Februari", "Maret", "April", "Mei", "Juni",
    "Juli", "Agustus", "September", "Oktober", "November", "Desember"
]

# Reuse exact URLs discovered in Cell 1 for Apr-Dec 2025
hist_lookup = {}
for _, row in df_ojk_hist.iterrows():
    t = row["title"].lower()
    for yr in (2025, 2024, 2023):
        for mo in MONTHS_ID:
            if mo.lower() in t and str(yr) in t:
                hist_lookup[(yr, mo)] = row["page_url"]

target_periods = [
    (2023, "Juni"), (2023, "Desember"),
    (2024, "Januari"), (2024, "Juni"), (2024, "September"), (2024, "Desember"),
] + [(2025, m) for m in MONTHS_ID]

slug_templates = [
    "Statistik-LPBBTI-{month}-{year}.aspx",
    "-Statistik-LPBBTI-{month}-{year}.aspx",
    "Statistik-P2P-Lending-Periode-{month}-{year}.aspx",
    "Statistik-Fintech-Lending-Periode-{month}-{year}.aspx",
]

base_pages_url = "https://ojk.go.id/id/kanal/iknb/data-dan-statistik/fintech/Pages/"
xlsx_records = []

for year, month in target_periods:
    m_idx = MONTHS_ID.index(month) + 1
    out_name = f"ojk_lpbbti_{year}_{m_idx:02d}.xlsx"
    out_path = DIRS["raw_ojk_xlsx"] / out_name

    if not (out_path.exists() and out_path.stat().st_size > 10_000):
        page_url = hist_lookup.get((year, month))
        if not page_url:
            for tmpl in slug_templates:
                candidate = base_pages_url + tmpl.format(month=month, year=year)
                r_head = http_sess.head(candidate, timeout=15)
                if r_head.status_code == 200:
                    page_url = candidate
                    break

        found_xlsx_url = None
        if page_url:
            r_pg = http_sess.get(page_url, timeout=25)
            if r_pg.status_code == 200:
                soup_pg = BeautifulSoup(r_pg.text, "lxml")
                for a in soup_pg.find_all("a", href=True):
                    if ".xlsx" in a["href"].lower():
                        href = a["href"]
                        found_xlsx_url = href if href.startswith("http") else "https://ojk.go.id" + href
                        break

        if not found_xlsx_url:
            xlsx_records.append({
                "year": year, "month": month, "status": "NOT_FOUND",
                "file_name": out_name, "bytes": 0, "n_sheets": 0
            })
            continue

        r_bin = http_sess.get(found_xlsx_url, timeout=60)
        r_bin.raise_for_status()
        out_path.write_bytes(r_bin.content)
        time.sleep(0.25)

    wb = openpyxl.load_workbook(out_path, read_only=True, data_only=True)
    n_sheets = len(wb.sheetnames)
    wb.close()

    xlsx_records.append({
        "year": year, "month": month, "status": "OK",
        "file_name": out_name, "bytes": out_path.stat().st_size, "n_sheets": n_sheets
    })

df_xlsx_inv = pd.DataFrame(xlsx_records)
print(f"ojk_xlsx_ready={(df_xlsx_inv['status'] == 'OK').sum()}/{len(df_xlsx_inv)} | total_kb={df_xlsx_inv['bytes'].sum() / 1024:.1f}")
print(df_xlsx_inv.to_string(index=False))

# 2. Download and parse OJK Direktori Industri LPBBTI Mei 2026 PDF
pdf_url = "https://ojk.go.id/id/kanal/iknb/data-dan-statistik/direktori/fintech/Documents/Direktori%20Industri%20LPBBTI%20-%20Mei%202026.pdf"
pdf_path = DIRS["raw_ojk_directory"] / "Direktori_Industri_LPBBTI_Mei_2026.pdf"
if not (pdf_path.exists() and pdf_path.stat().st_size > 10_000):
    r_pdf = http_sess.get(pdf_url, timeout=60)
    r_pdf.raise_for_status()
    pdf_path.write_bytes(r_pdf.content)

raw_rows = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        for tbl in page.extract_tables():
            raw_rows.extend(tbl)

data_rows = [r for r in raw_rows if r and r[0] and str(r[0]).strip().isdigit()]
raw_cols = [
    "no", "kategori_lembaga", "nama_perusahaan", "nama_sistem_elektronik",
    "nomor_izin", "tanggal_izin_raw", "jenis_usaha", "wilayah_kerja",
    "alamat_website", "sistem_operasi"
]
df_dir_raw = pd.DataFrame(data_rows, columns=raw_cols)
dir_csv = DIRS["raw_ojk_directory"] / "ojk_direktori_lpbbti_mei2026_raw.csv"
df_dir_raw.to_csv(dir_csv, index=False)

print(f"ojk_directory_pdf_bytes={pdf_path.stat().st_size} | parsed_platforms={len(df_dir_raw)}")
print("jenis_usaha_counts=", df_dir_raw["jenis_usaha"].value_counts(dropna=False).to_dict())
print("sistem_operasi_counts=", df_dir_raw["sistem_operasi"].value_counts(dropna=False).to_dict())
print(df_dir_raw.head(5)[["no", "nama_perusahaan", "nama_sistem_elektronik", "tanggal_izin_raw", "jenis_usaha", "sistem_operasi"]].to_string(index=False))


ojk_xlsx_ready=18/18 | total_kb=7201.2
 year     month status               file_name  bytes  n_sheets
 2023      Juni     OK ojk_lpbbti_2023_06.xlsx 393133        22
 2023  Desember     OK ojk_lpbbti_2023_12.xlsx 542276        24
 2024   Januari     OK ojk_lpbbti_2024_01.xlsx 532191        25
 2024      Juni     OK ojk_lpbbti_2024_06.xlsx 417402        24
 2024 September     OK ojk_lpbbti_2024_09.xlsx 391116        24
 2024  Desember     OK ojk_lpbbti_2024_12.xlsx 390605        24
 2025   Januari     OK ojk_lpbbti_2025_01.xlsx 390633        24
 2025  Februari     OK ojk_lpbbti_2025_02.xlsx 390441        24
 2025     Maret     OK ojk_lpbbti_2025_03.xlsx 390861        24
 2025     April     OK ojk_lpbbti_2025_04.xlsx 392011        24
 2025       Mei     OK ojk_lpbbti_2025_05.xlsx 390873        24
 2025      Juni     OK ojk_lpbbti_2025_06.xlsx 395294        24
 2025      Juli     OK ojk_lpbbti_2025_07.xlsx 391107        24
 2025   Agustus     OK ojk_lpbbti_2025_08.xlsx 392898        24
 

ojk_directory_pdf_bytes=301203 | parsed_platforms=94
jenis_usaha_counts= {'Konvensional': 87, 'Syariah': 7}
sistem_operasi_counts= {'Android': 37, 'Android & iOS': 32, '-': 24, 'Andorid': 1}
no              nama_perusahaan nama_sistem_elektronik tanggal_izin_raw  jenis_usaha sistem_operasi
 1       PT Pasar Dana Pinjaman                Danamas     06 Juli 2017 Konvensional        Android
 2      PT Amartha Mikro Fintek                Amartha      13 Mei 2019 Konvensional        Android
 3              PT Indo Fin Tek           Dompet Kilat      13 Mei 2019 Konvensional        Android
 4 PT Creative Mobile Adventure                  Boost      13 Mei 2019 Konvensional              -
 5    PT Toko Modal Mitra Usaha              Tokomodal      24 Mei 2019 Konvensional        Android


In [4]:
# Cell 4 - Scrape BPS provincial socio-economic tables (2023-2026) via RSC + Next-Action API
import itertools
import json
from urllib.parse import urlparse

BPS_NEXT_ACTION_ID = "401007016a731a34d67075973e6650422e93d71814"

def decode_bps_rsc(html_text):
    chunks = re.findall(r'self\.__next_f\.push\(\[1,"(.*?)"\]\)', html_text, flags=re.DOTALL)
    out = []
    for c in chunks:
        try:
            out.append(json.loads('"' + c + '"'))
        except Exception:
            pass
    return "".join(out)

def json_obj_to_rows_type2(obj, indicator_slug):
    dc = obj.get("datacontent") or {}
    if not dc:
        return pd.DataFrame()
    rows = []
    for vv, vr, tv, th, tt in itertools.product(
        obj.get("vervar", []),
        obj.get("var", []),
        obj.get("turvar", []),
        obj.get("tahun", []),
        obj.get("turtahun", []),
    ):
        key = f"{vv['val']}{vr['val']}{tv['val']}{th['val']}{tt['val']}"
        if key in dc:
            rows.append({
                "indicator": indicator_slug,
                "kode_prov": vv["val"],
                "provinsi_raw": vv["label"],
                "variabel": vr["label"],
                "satuan": vr.get("unit", ""),
                "turunan_var": tv["label"],
                "tahun": th["label"],
                "periode": tt["label"],
                "nilai_raw": dc[key],
            })
    return pd.DataFrame(rows)

def fetch_bps_type2_multiyr(base_url, table_id_b64, indicator_slug, target_years=("2023", "2024", "2025", "2026")):
    r0 = http_sess.get(base_url, timeout=25)
    r0.raise_for_status()
    dec = decode_bps_rsc(r0.text)

    # Extract initial SSR table object
    frames = []
    idx_dc = dec.find('"datacontent":')
    ssr_years = set()
    if idx_dc != -1:
        start_obj = dec.rfind('{"status":', 0, idx_dc)
        if start_obj == -1:
            start_obj = dec.rfind('{"var":', 0, idx_dc)
        if start_obj != -1:
            obj0, _ = json.JSONDecoder().raw_decode(dec[start_obj:])
            df0 = json_obj_to_rows_type2(obj0, indicator_slug)
            if not df0.empty:
                frames.append(df0)
                ssr_years.update(df0["tahun"].astype(str).unique())

    # Extract listTahun and query remaining target years via Next-Action Server Action
    idx_lt = dec.find('"listTahun":[')
    if idx_lt != -1:
        lt, _ = json.JSONDecoder().raw_decode(dec[idx_lt + len('"listTahun":'):])
        parsed = urlparse(base_url)
        origin = f"{parsed.scheme}://{parsed.netloc}"
        for item in lt:
            yr_str = str(item.get("th", ""))
            if yr_str in target_years and yr_str not in ssr_years:
                rp = http_sess.post(
                    base_url,
                    headers={
                        "Accept": "text/x-component",
                        "Next-Action": BPS_NEXT_ACTION_ID,
                        "Content-Type": "text/plain;charset=UTF-8",
                        "Origin": origin,
                        "Referer": base_url,
                    },
                    data=json.dumps([{"locale": "id", "id": table_id_b64, "year": str(item["th_id"])}]),
                    timeout=25,
                )
                for ln in rp.text.splitlines():
                    if '"datacontent"' in ln and ":" in ln:
                        payload = json.loads(ln.split(":", 1)[1])
                        resp_obj = payload.get("response") or {}
                        df_yr = json_obj_to_rows_type2(resp_obj, indicator_slug)
                        if not df_yr.empty:
                            frames.append(df_yr)
                        break
                time.sleep(0.15)

    return pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()

def parse_bps_type3(html_text, indicator_slug, req_year):
    dec = decode_bps_rsc(html_text)
    idx_k = dec.find('"kolom":')
    if idx_k == -1:
        return pd.DataFrame(), None
    decoder = json.JSONDecoder()
    pos = dec.rfind("{", 0, idx_k)
    obj = None
    while pos != -1:
        try:
            cand, _ = decoder.raw_decode(dec[pos:])
            if isinstance(cand, dict) and "kolom" in cand and "data" in cand:
                obj = cand
                break
        except Exception:
            pass
        pos = dec.rfind("{", 0, pos)
    if not obj or not obj.get("data"):
        return pd.DataFrame(), None

    actual_year = obj.get("tahun_data", req_year)
    kolom_meta = obj.get("kolom", {})
    rows = []
    for item in obj["data"]:
        prov_label = item.get("label") or item.get("label_raw")
        kode_wil = item.get("kode_wilayah")
        variables = item.get("variables") or {}
        for var_id, var_val in variables.items():
            meta = kolom_meta.get(var_id, {})
            rows.append({
                "indicator": indicator_slug,
                "kode_wilayah": kode_wil,
                "provinsi_raw": prov_label,
                "tahun": actual_year,
                "variabel": meta.get("nama_variabel", var_id),
                "satuan": meta.get("satuan", ""),
                "nilai_raw": var_val.get("value_raw") if isinstance(var_val, dict) else var_val,
            })
    return pd.DataFrame(rows), actual_year

BPS_TYPE2_SPECS = [
    ("tpt", "NTQzIzI=", "https://www.bps.go.id/id/statistics-table/2/NTQzIzI=/tingkat-pengangguran-terbuka-menurut-provinsi.html"),
    ("informal", "MjE1MyMy", "https://www.bps.go.id/id/statistics-table/2/MjE1MyMy/proporsi-lapangan-kerja-informal-menurut-provinsi.html"),
    ("upah_jam", "MTE3MiMy", "https://www.bps.go.id/id/statistics-table/2/MTE3MiMy/upah-rata---rata-per-jam-pekerja-menurut-provinsi.html"),
    ("upah_bersih", "NTgzIzI=", "https://www.bps.go.id/id/statistics-table/2/NTgzIzI=/rata-rata-upah-gaji-bersih-sebulan-buruh-karyawan-pegawai-menurut-provinsi-dan-jenis-pekerjaan-utama.html"),
    ("ipm", "NDEzIzI=", "https://www.bps.go.id/id/statistics-table/2/NDEzIzI=/-metode-baru--indeks-pembangunan-manusia.html"),
    ("ump", "MjgyNCMy", "https://jateng.bps.go.id/id/statistics-table/2/MjgyNCMy/upah-minimum-provinsi--ump--per-bulan-menurut-provinsi-di-indonesia.html"),
]

BPS_TYPE3_SPECS = [
    ("kemiskinan", "https://www.bps.go.id/id/statistics-table/3/UkVkWGJVZFNWakl6VWxKVFQwWjVWeTlSZDNabVFUMDkjMw==/jumlah-dan-persentase-penduduk-miskin-menurut-provinsi--2024.html", [2023, 2024, 2025, 2026]),
    ("pengeluaran", "https://www.bps.go.id/id/statistics-table/3/V1ZKMWVrSTNOek5ZZUZOcVZEZGFValJvV0hWalFUMDkjMw==/rata-rata-pengeluaran-per-kapita-sebulan-makanan-dan-bukan-makanan-di-daerah-perkotaan-dan-perdesaan-menurut-provinsi--rupiah-.html", [2023, 2024, 2025, 2026]),
    ("penduduk", "https://www.bps.go.id/id/statistics-table/3/V1ZSbFRUY3lTbFpEYTNsVWNGcDZjek53YkhsNFFUMDkjMw==/penduduk--laju-pertumbuhan-penduduk--distribusi-persentase-penduduk--kepadatan-penduduk--rasio-jenis-kelamin-penduduk-menurut-provinsi--2024.html", [2023, 2024, 2025, 2026]),
    ("pdrb_perkapita", "https://www.bps.go.id/id/statistics-table/3/YWtoQlRVZzNiMU5qU1VOSlRFeFZiRTR4VDJOTVVUMDkjMw==/produk-domestik-regional-bruto-per-kapita-atas-dasar-harga-berlaku-menurut-provinsi--ribu-rupiah---2022.html", [2023, 2024, 2025, 2026]),
]

bps_audit = []

for slug, tid, base_url in BPS_TYPE2_SPECS:
    out_p = DIRS["raw_bps"] / f"bps_{slug}_raw.csv"
    if not FORCE_REFRESH and out_p.exists() and out_p.stat().st_size > 100:
        df_all = pd.read_csv(out_p)
    else:
        df_all = fetch_bps_type2_multiyr(base_url, tid, slug)
        if not df_all.empty:
            df_all.to_csv(out_p, index=False)
    if not df_all.empty:
        yrs_found = sorted(df_all["tahun"].astype(str).unique())
        bps_audit.append({
            "slug": slug, "type": "type2", "rows": len(df_all),
            "n_prov": df_all["provinsi_raw"].nunique(),
            "years": ",".join(yrs_found), "file": out_p.name
        })

for slug, base_url, years in BPS_TYPE3_SPECS:
    out_p = DIRS["raw_bps"] / f"bps_{slug}_raw.csv"
    if not FORCE_REFRESH and out_p.exists() and out_p.stat().st_size > 100:
        df_all = pd.read_csv(out_p)
    else:
        frames = []
        seen_years = set()
        for yr in years:
            r_bps = http_sess.get(base_url, params={"year": yr}, timeout=30)
            if r_bps.status_code == 200:
                df_yr, actual_yr = parse_bps_type3(r_bps.text, slug, yr)
                if not df_yr.empty and str(actual_yr) not in seen_years:
                    seen_years.add(str(actual_yr))
                    frames.append(df_yr)
            time.sleep(0.15)
        df_all = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()
        if not df_all.empty:
            df_all.to_csv(out_p, index=False)
    if not df_all.empty:
        yrs_found = sorted(df_all["tahun"].astype(str).unique())
        bps_audit.append({
            "slug": slug, "type": "type3", "rows": len(df_all),
            "n_prov": df_all["provinsi_raw"].nunique(),
            "years": ",".join(yrs_found), "file": out_p.name
        })

ump_ref_path = DIRS["raw_bps"] / "refnusa_ump_1997_2026_raw.csv"
if FORCE_REFRESH or not (ump_ref_path.exists() and ump_ref_path.stat().st_size > 1000):
    r_ump_ref = http_sess.get("https://refnusa.dev/data/upah-minimum-provinsi.csv", timeout=30)
    r_ump_ref.raise_for_status()
    ump_ref_path.write_bytes(r_ump_ref.content)
df_ump_ref = pd.read_csv(ump_ref_path)
bps_audit.append({
    "slug": "ump_refnusa", "type": "csv", "rows": len(df_ump_ref),
    "n_prov": df_ump_ref["province_name"].nunique(),
    "years": f"{df_ump_ref['year'].min()}-{df_ump_ref['year'].max()}",
    "file": ump_ref_path.name
})

df_bps_inv = pd.DataFrame(bps_audit)
print(f"bps_sources_extracted={len(df_bps_inv)} | total_rows={df_bps_inv['rows'].sum()}")
print(df_bps_inv.to_string(index=False))


bps_sources_extracted=11 | total_rows=6314
          slug  type  rows  n_prov               years                          file
           tpt type2   304      39 2023,2024,2025,2026               bps_tpt_raw.csv
      informal type2   113      39      2023,2024,2025          bps_informal_raw.csv
      upah_jam type2   113      39      2023,2024,2025          bps_upah_jam_raw.csv
   upah_bersih type2  1733      39      2024,2025,2026       bps_upah_bersih_raw.csv
           ipm type2  1132     553           2023,2024               bps_ipm_raw.csv
           ump type2   147      38 2023,2024,2025,2026               bps_ump_raw.csv
    kemiskinan type3   567      39      2023,2024,2025        bps_kemiskinan_raw.csv
   pengeluaran type3   339      39      2023,2024,2025       bps_pengeluaran_raw.csv
      penduduk type3   780      39 2023,2024,2025,2026          bps_penduduk_raw.csv
pdrb_perkapita type3   114      38      2023,2024,2025    bps_pdrb_perkapita_raw.csv
   ump_refnusa   csv  

In [5]:
# Cell 5 - Scrape Google Play Store apps/reviews for licensed Pindar & OJK Satgas PASTI data
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus
from google_play_scraper import app as gp_app, reviews as gp_reviews, Sort

STOP_TOKENS = {"pt", "tbk", "indonesia", "finansial", "fintech", "teknologi", "digital", "keuangan", "solusi", "mikro", "dana", "kredit", "modal", "pinjaman", "group", "asia", "nusantara"}

def norm_tokens(text):
    cleaned = re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()
    return [t for t in cleaned.split() if len(t) >= 3 and t not in STOP_TOKENS]

def find_playstore_match(row_dict):
    brand = str(row_dict["nama_sistem_elektronik"]).strip()
    comp = str(row_dict["nama_perusahaan"]).strip()
    brand_clean = re.sub(r"[^a-z0-9]+", "", brand.lower())
    comp_tokens = set(norm_tokens(comp))

    queries = [f"{brand} {comp}", brand]
    candidate_pkgs = []
    for q in queries:
        try:
            url_q = f"https://play.google.com/store/search?q={quote_plus(q)}&c=apps&hl=id&gl=id"
            r_ps = http_sess.get(url_q, timeout=15)
            if r_ps.status_code == 200:
                pkgs = re.findall(r"/store/apps/details\?id=([a-zA-Z0-9._]+)", r_ps.text)
                for p in pkgs:
                    if p not in candidate_pkgs:
                        candidate_pkgs.append(p)
        except Exception:
            pass
        if len(candidate_pkgs) >= 5:
            break

    for pkg in candidate_pkgs[:5]:
        try:
            info = gp_app(pkg, lang="id", country="id")
        except Exception:
            continue
        title_str = str(info.get("title", ""))
        dev_str = str(info.get("developer", ""))
        title_clean = re.sub(r"[^a-z0-9]+", "", title_str.lower())
        dev_tokens = set(norm_tokens(dev_str))

        comp_overlap = len(comp_tokens & dev_tokens) >= 1 and len(comp_tokens) > 0
        brand_in_title = len(brand_clean) >= 4 and brand_clean in title_clean
        pkg_match = len(brand_clean) >= 5 and brand_clean in pkg.lower().replace(".", "")

        if comp_overlap or brand_in_title or pkg_match:
            return {
                "no_ojk": row_dict["no"],
                "nama_perusahaan": comp,
                "nama_sistem_elektronik": brand,
                "jenis_usaha": row_dict["jenis_usaha"],
                "sistem_operasi_ojk": row_dict["sistem_operasi"],
                "appId": info.get("appId"),
                "title": title_str,
                "developer": dev_str,
                "score": info.get("score"),
                "ratings": info.get("ratings"),
                "reviews_total": info.get("reviews"),
                "installs": info.get("installs"),
                "realInstalls": info.get("realInstalls"),
                "histogram": str(info.get("histogram")),
                "released": info.get("released"),
                "updated": info.get("updated"),
                "genre": info.get("genre"),
                "match_rule": "comp_dev" if comp_overlap else ("title_brand" if brand_in_title else "pkg_brand"),
            }
    return None

apps_csv = DIRS["raw_playstore"] / "playstore_pindar_apps_raw.csv"
if not FORCE_REFRESH and apps_csv.exists() and apps_csv.stat().st_size > 100:
    df_gp_apps = pd.read_csv(apps_csv)
else:
    platform_records = df_dir_raw.to_dict("records")
    matched_apps = []
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = [ex.submit(find_playstore_match, rec) for rec in platform_records]
        for fut in as_completed(futures):
            res = fut.result()
            if res is not None:
                matched_apps.append(res)
    df_gp_apps = pd.DataFrame(matched_apps).sort_values("no_ojk").reset_index(drop=True)
    df_gp_apps.to_csv(apps_csv, index=False)

print(f"playstore_matched_apps={len(df_gp_apps)}/{len(df_dir_raw)} | jenis_usaha={df_gp_apps['jenis_usaha'].value_counts().to_dict()}")
print(df_gp_apps.head(8)[["no_ojk", "nama_sistem_elektronik", "appId", "score", "realInstalls", "reviews_total", "match_rule"]].to_string(index=False))

def fetch_app_reviews(app_row):
    pkg = app_row["appId"]
    try:
        rvs, _ = gp_reviews(pkg, lang="id", country="id", sort=Sort.NEWEST, count=60)
    except Exception:
        return []
    out = []
    for rv in rvs:
        out.append({
            "no_ojk": app_row["no_ojk"],
            "nama_sistem_elektronik": app_row["nama_sistem_elektronik"],
            "nama_perusahaan": app_row["nama_perusahaan"],
            "jenis_usaha": app_row["jenis_usaha"],
            "appId": pkg,
            "reviewId": rv.get("reviewId"),
            "userName": rv.get("userName"),
            "score": rv.get("score"),
            "thumbsUpCount": rv.get("thumbsUpCount"),
            "at": rv.get("at"),
            "appVersion": rv.get("appVersion"),
            "content": rv.get("content"),
            "replyContent": rv.get("replyContent"),
        })
    return out

reviews_csv = DIRS["raw_playstore"] / "playstore_pindar_reviews_raw.csv"
if not FORCE_REFRESH and reviews_csv.exists() and reviews_csv.stat().st_size > 100:
    df_gp_reviews = pd.read_csv(reviews_csv)
else:
    active_review_apps = df_gp_apps[df_gp_apps["reviews_total"].fillna(0) > 0].to_dict("records")
    all_reviews = []
    with ThreadPoolExecutor(max_workers=4) as ex:
        for r_list in ex.map(fetch_app_reviews, active_review_apps):
            all_reviews.extend(r_list)
    df_gp_reviews = pd.DataFrame(all_reviews)
    df_gp_reviews.to_csv(reviews_csv, index=False)

print(f"playstore_reviews_scraped={len(df_gp_reviews)} | apps_with_reviews={df_gp_reviews['appId'].nunique()} | score_dist={df_gp_reviews['score'].value_counts().sort_index().to_dict()}")

# 2. Download and parse Satgas PASTI Illegal Pinjol PDF (Oktober 2026)
illegal_pdf_url = "https://ojk.go.id/id/berita-dan-kegiatan/info-terkini/Documents/Pages/Satgas-PASTI-Tindak-Aktivitas-Keuangan-Ilegal%2c-Penipuan-Transaksi-Keuangan%2c-dan-Pergadaian-Ilegal/Lampiran%20Daftar%20Website%20Pinjaman%20Online%20Ilegal%20Oktober%202026.pdf"
illegal_pdf_path = DIRS["raw_playstore"] / "Lampiran_Pinjaman_Online_Ilegal_Oktober_2026.pdf"
if FORCE_REFRESH or not (illegal_pdf_path.exists() and illegal_pdf_path.stat().st_size > 10_000):
    r_ill = http_sess.get(illegal_pdf_url, timeout=60)
    r_ill.raise_for_status()
    illegal_pdf_path.write_bytes(r_ill.content)

ill_rows = []
with pdfplumber.open(illegal_pdf_path) as pdf:
    for page in pdf.pages:
        for tbl in page.extract_tables():
            for row in tbl:
                if row and row[0] and str(row[0]).strip().isdigit():
                    ill_rows.append({
                        "nomor": int(str(row[0]).strip()),
                        "url": row[1] if len(row) > 1 else None,
                        "nama_entitas": row[2] if len(row) > 2 else None,
                        "developer": row[3] if len(row) > 3 else None,
                    })

df_illegal = pd.DataFrame(ill_rows)
illegal_csv = DIRS["raw_playstore"] / "satgas_pasti_pinjol_ilegal_okt2026_raw.csv"
df_illegal.to_csv(illegal_csv, index=False)
print(f"satgas_pasti_illegal_pinjol_rows={len(df_illegal)}")

# 3. Scrape Satgas PASTI Siaran Pers index pages 1..14
satgas_sp_csv = DIRS["raw_playstore"] / "satgas_pasti_siaran_pers_raw.csv"
if not FORCE_REFRESH and satgas_sp_csv.exists() and satgas_sp_csv.stat().st_size > 100:
    df_satgas_sp = pd.read_csv(satgas_sp_csv)
else:
    sp_rows = []
    for pg in range(1, 15):
        r_sp = http_sess.get(f"https://satgaspasti.ojk.go.id/SiaranPers/Index?page={pg}", timeout=25)
        if r_sp.status_code != 200:
            break
        soup_sp = BeautifulSoup(r_sp.text, "lxml")
        cards = soup_sp.find_all("a", href=lambda h: h and "/SiaranPers/Detail" in h)
        if not cards:
            break
        for card in cards:
            title_el = card.select_one(".article-title")
            date_el = card.select_one(".article-date")
            content_el = card.select_one(".article-content")
            sp_rows.append({
                "page": pg,
                "detail_url": "https://satgaspasti.ojk.go.id" + card["href"],
                "title": title_el.get_text(" ", strip=True) if title_el else card.get_text(" ", strip=True),
                "date_raw": date_el.get_text(" ", strip=True) if date_el else None,
                "summary": content_el.get_text(" ", strip=True) if content_el else None,
            })
        time.sleep(0.15)
    df_satgas_sp = pd.DataFrame(sp_rows).drop_duplicates(subset=["detail_url"])
    df_satgas_sp.to_csv(satgas_sp_csv, index=False)

print(f"satgas_pasti_press_releases={len(df_satgas_sp)}")
print(df_satgas_sp.head(5)[["date_raw", "title"]].to_string(index=False))


playstore_matched_apps=72/94 | jenis_usaha={'Konvensional': 66, 'Syariah': 6}
no_ojk nama_sistem_elektronik                   appId    score  realInstalls  reviews_total match_rule
     1                Danamas  com.danamas.mergingapp 2.140000        338786          232.0   comp_dev
    10                KlikA2C         com.klika2c.app 1.920000        113746          142.0   comp_dev
    11              Akseleran com.akseleran.crowdfund 3.295620        876352         7890.0   comp_dev
    12                 Ammana        com.alami_funder 3.716561        305419          878.0   comp_dev
    13             PinjamanGo          com.pinjamango 4.589085       9976200       268653.0   comp_dev
    14                KoinP2P       com.koinworks.app 3.439182       2647626        15650.0   comp_dev
    15              Pohondana       com.pdi.pohondana 2.333333         41021           21.0   comp_dev
    16                  Mekar        com.mobile.mekar      NaN          4769            NaN   comp

playstore_reviews_scraped=4061 | apps_with_reviews=66 | score_dist={1: 1323, 2: 159, 3: 154, 4: 198, 5: 2227}


satgas_pasti_illegal_pinjol_rows=330


satgas_pasti_press_releases=79
        date_raw                                                                                                                                                            title
15 November 2025                                                                              SATGAS PASTI IMBAU MASYARAKAT WASPADAI PENIPUAN MENGGUNAKAN ARTIFICIAL INTELLIGENCE
 15 Oktober 2025                   Satgas PASTI melalui Indonesia Anti-Scam Centre (IASC) bersama Polda Sumatera Utara Tangkap Pelaku Penipuan Keuangan yang Merugikan Masyarakat
 13 Oktober 2025 Satgas PASTI Hentikan Kegiatan Golden Eagle International – UNDP (Golden Eagle) yang Menawarkan Program Penghapusan Utang dan Pembiayaan Investasi Non APBN/APBD
    16 Juli 2025                                                                                                   Satgas PASTI Hentikan Kegiatan Usaha Omnicom Group (OMC) Palsu
    19 Juni 2025                                              Satgas PASTI Blok